# Mission 2 L2(CNN) — 처음부터 다시 학습 (원본 데이터 → 스펙트로그램 → 학습 → 앙상블)

멘토 답변(2026-10-05) 반영:
1. **처음부터 재학습** — Drive의 원본 zip에서 음성을 풀고, 스펙트로그램을 새로 만들고, 학습까지 이 노트북 하나로 한다.
   예전 캐시(tar, 따로 복원한 인덱스)는 쓰지 않는다. 라벨은 제공된 JSON의 `speaker` 그대로(`label_source=provided_json`).
2. **에폭별 로그 전체 보존** — 모든 출력이 셀과 Drive의 로그 파일(`rerun_log_*.txt`) 양쪽에 남는다.
   마지막 셀이 로그 파일 전체를 다시 출력하므로, 런타임이 끊겼다 이어서 돌린 경우에도 제출용 노트북에 전체 기록이 남는다.
   **제출 전에 출력 지우기 금지.**
3. **제출 코드만으로 재현** — 이전 세션이나 다른 계정이 만든 파일은 쓰지 않는다. 실행마다 `RUN_ID` 폴더를 새로 만들고,
   그 안에서 이번 실행이 만든 것만 쓴다. 코드는 제출 코드 폴더(`CODE_DIR`)를 쓰고, 비워 두면 GitHub 브랜치
   `gwanghee/mission2-l2-fast-rerun`(PR #43)을 받는다. 어느 쪽이든 코드 지문(파일 해시)을 로그에 남긴다.

학습 방식은 91.55% 모델과 같다 (SpeakerCNNLarge, 40 mel, 배치 64, Adam 1e-3, ReduceLROnPlateau, `--seed 42` 분할,
`--torch-seed 1/2/3`, 3개 소프트보팅). 바뀐 것은 **속도와 멈추는 시점**뿐이다:

| | 이전 | 이번 |
|---|---|---|
| 스펙트로그램 읽기 | 발화마다 .npy 파일 하나씩 (배치당 ~35ms, 메인 스레드) | 파일 하나로 묶어 메모리 매핑 (배치당 ~2ms, 워커 2개) |
| GPU 연산 | float32 | 혼합 정밀도(`--amp`, T4 텐서코어) |
| 에폭 수 | 30 고정 | 최대 30, **5에폭 연속 개선 없으면 자동 멈춤** |
| 저장 단위 | 5에폭마다 | **1에폭마다** (끊겨도 최대 1에폭만 손실, 이어 학습해도 결과 동일 — 테스트로 확인) |

### 사용법
- **Colab Pro**, GPU 런타임(T4 이상), 계정 하나, 이 노트북 하나로 시드 3개를 차례로 모두 돌린다. 예상 10~12시간
  (첫 에폭 로그의 `time=…s`로 확인). **런타임 > 모두 실행**으로 시작하고 브라우저 탭은 켜 둔다.
- 새로 처음부터 돌릴 때는 1번 셀의 `RUN_ID`를 **새 이름**으로 바꾼다(예: 날짜). 같은 `RUN_ID` 폴더가 있으면
  그 실행을 이어서 하는 것으로 본다.
- 런타임이 끊기면 `RUN_ID`를 **그대로 두고** 다시 "모두 실행" → 같은 실행이 만든 데이터와 마지막 에폭부터 이어간다
  (이어 학습해도 끊김 없이 학습한 것과 가중치가 같음 — 테스트로 확인). 끊김과 이어간 기록도 로그에 남는다.

## 실행 기록 안내 (심사용)

- **로그 시각은 UTC입니다** (한국 시간 = +9시간). 마지막 셀이 Drive의 로그 파일(`rerun_log.txt`)을 처음부터 끝까지 출력하므로, 전체 실행 기록(2026-10-05 데이터 생성부터)은 **마지막 셀 출력**에서 볼 수 있습니다.
- **이어서 학습**: Colab 런타임이 끊기면 같은 `RUN_ID`로 다시 실행해 Drive에 저장된 마지막 에폭부터 이어서 학습했습니다(옵티마이저·학습률·난수 상태까지 이어받음). 끊긴 시점은 로그의 "세션 재시작 -- 같은 실행을 이어감" 줄로 남아 있습니다.
- **두 번 나오는 에폭**: ts2의 7번째, ts3의 9번째 에폭은 학습이 끝난 직후 Drive에 저장되기 전에 런타임이 끊겨, 직전 저장 지점부터 같은 에폭을 다시 학습했습니다. 그래서 로그에 두 번 나타나며, 최종 모델에는 다시 학습한 쪽이 반영됐습니다. 빠진 에폭은 없습니다(학습 66에폭 = ts1 22 + ts2 24 + ts3 20).
- **이 노트북 파일의 셀 출력**: 학습을 마친 세션의 노트북 파일이 저장되지 않아, 2026-10-12에 같은 노트북을 같은 `RUN_ID`로 다시 실행했습니다. 학습은 이미 끝나 건너뛰었고(셀 4), 같은 체크포인트로 앙상블을 다시 묶어 같은 sha256(`19ea482a…3246`)을 확인했으며(셀 5), 공식 Validation을 다시 평가해 같은 결과(91.4888%)를 얻었습니다(셀 6).
- **로그의 FAILED 2건**: 둘 다 공식 Validation 평가 단계에서 Validation 폴더 경로를 잘못 찾은 것으로(노트북 버그, 이후 수정), 학습과는 무관합니다. 각각 바로 뒤에 성공한 평가 기록이 있습니다.


In [1]:
# 1. 설정 -- 새로 처음부터 돌릴 때 RUN_ID 만 새 이름으로 바꾼다 (끊겨서 이어갈 때는 그대로 둔다)
RUN_ID = "rerun_2026-10-06b"
SEEDS = [1, 2, 3]              # 앙상블 3개 = torch seed 1/2/3 (91.55% 모델과 같음, 바꾸지 말 것)
MAX_EPOCHS = 30
EARLY_STOP_PATIENCE = 5        # 최고 val_acc 이후 5에폭 연속 개선 없으면 멈춤
BRANCH = "gwanghee/mission2-l2-fast-rerun"  # PR #43
REPO = "github.com/eigensinn-dcc2026/prelim.git"

DRIVE_ROOT = "/content/drive/MyDrive/데이터 크리에이터 캠프"
CODE_DIR = f"{DRIVE_ROOT}/제출/mission2"   # 제출 코드 폴더 (inference.py, scripts/, src/, requirements.txt). "" 이면 GitHub 에서 받음
ZIP_DIR = f"{DRIVE_ROOT}/대학부 데이터"                 # 원본 zip 이 있는 곳
OUT_DIR = f"{DRIVE_ROOT}/m2_cache/m2_rerun_from_raw/{RUN_ID}"   # 이번 실행의 모든 결과(데이터, 체크포인트, 로그)

DATA_DIR = "/content/dcc_data_full"       # 원본 압축을 푸는 곳 (Colab 로컬 디스크)
SPEC_DIR = "/content/m2_specs"            # 발화별 스펙트로그램 (묶은 뒤 지움)
PACKED_NAME = "m2_spectrograms_packed.npy"
PACKED_INDEX_NAME = "m2_spectrogram_index_packed.parquet"
LOCAL_CKPT_DIR = "/content/ckpt"

In [2]:
# 2. Drive 마운트 + GPU 확인 + 로그 도구
import datetime, hashlib, os, shutil, subprocess, sys

from google.colab import drive
drive.mount("/content/drive")
CONTINUING = os.path.isdir(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
LOG_FILE = f"{OUT_DIR}/rerun_log.txt"


def log(message: str) -> None:
    line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {message}"
    print(line, flush=True)
    with open(LOG_FILE, "a", encoding="utf-8") as stream:
        stream.write(line + "\n")


def run(cmd: str, *, cwd: str = "/content/prelim", shown: str | None = None) -> None:
    """명령을 실행하면서 출력을 한 줄씩 셀과 로그 파일에 동시에 남긴다 (os.system 은 셀에 출력이 안 남았다)."""
    log(f"$ {shown or cmd}")
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "60"}  # 진행 막대는 1분에 한 줄만
    process = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=env)
    with open(LOG_FILE, "a", encoding="utf-8") as stream:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
    status = process.wait()
    if status != 0:
        log(f"FAILED (exit {status})")
        raise RuntimeError(f"명령 실패 (exit {status}): {shown or cmd}")


def copy_atomic(src: str, dst: str) -> None:
    """복사 도중 끊겨도 반쯤 쓴 파일이 남지 않게: 임시 이름으로 다 쓴 다음 이름을 바꾼다."""
    tmp = dst + ".tmp"
    shutil.copyfile(src, tmp)
    os.replace(tmp, dst)


def sha256(path: str) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


import torch
assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > T4 GPU 로 바꾸고 다시 시작할 것"
log("=" * 80)
log(f"{'세션 재시작 -- 같은 실행을 이어감' if CONTINUING else '새 실행 시작'}: RUN_ID={RUN_ID} SEEDS={SEEDS} "
    f"MAX_EPOCHS={MAX_EPOCHS} EARLY_STOP_PATIENCE={EARLY_STOP_PATIENCE}")
log(f"GPU={torch.cuda.get_device_name(0)} torch={torch.__version__} CPU={os.cpu_count()}개 "
    f"디스크 여유={shutil.disk_usage('/content').free / 1e9:.0f}GB")

Mounted at /content/drive
[2026-10-12 02:18:44] ================================================================================
[2026-10-12 02:18:45] 세션 재시작 -- 같은 실행을 이어감: RUN_ID=rerun_2026-10-06b SEEDS=[1, 2, 3] MAX_EPOCHS=30 EARLY_STOP_PATIENCE=5
[2026-10-12 02:18:45] GPU=Tesla T4 torch=2.11.0+cu130 CPU=2개 디스크 여유=208GB


In [3]:
# 3-1. 코드 준비: 제출 코드 폴더(CODE_DIR)가 있으면 그걸, 없으면 GitHub 브랜치를 받는다 (토큰은 로그에 남기지 않는다)
from getpass import getpass

if os.path.isdir("/content/prelim"):
    pass  # 같은 런타임에서 셀을 다시 실행한 경우
elif CODE_DIR:
    assert os.path.isfile(f"{CODE_DIR}/scripts/train_mission2_cnn.py"), f"CODE_DIR 에 코드가 없음: {CODE_DIR}"
    shutil.copytree(CODE_DIR, "/content/prelim")
    log(f"코드: 제출 코드 폴더 {CODE_DIR}")
else:
    token = getpass("GitHub PAT: ").strip()
    result = subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", f"https://x-access-token:{token}@{REPO}", "/content/prelim"],
        capture_output=True, text=True,
    )
    del token
    if result.returncode != 0:
        raise RuntimeError("clone 실패 -- 토큰이 만료됐거나 repo 권한이 없음:\n" + result.stderr.replace("x-access-token:", "")[-500:])
    # git 은 clone 한 주소를 .git/config 에 그대로 저장한다 -- 토큰이 파일에 남지 않게 주소에서 지운다
    subprocess.run(["git", "remote", "set-url", "origin", f"https://{REPO}"], cwd="/content/prelim", check=True)
    log(f"코드: GitHub {BRANCH}")
run("pip install -q -r requirements.txt")
run("pip list 2>/dev/null | grep -i -E '^(numpy|pandas|pyarrow|librosa|scikit-learn|lightgbm|tqdm|torch) '")  # 실행 환경 버전 기록
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd="/content/prelim", capture_output=True, text=True).stdout.strip()
# 코드 지문: 코드 폴더 안 .py 파일 전체를 경로 순서대로 해시 -- 제출한 코드와 같은 코드로 돌렸는지 대조용
_code_digest = hashlib.sha256()
for _dir, _subdirs, _files in sorted(os.walk("/content/prelim")):  # 폴더 경로 순서
    if "/." in _dir or "__pycache__" in _dir:
        continue
    for _name in sorted(f for f in _files if f.endswith(".py")):
        _path = os.path.join(_dir, _name)
        _code_digest.update(os.path.relpath(_path, "/content/prelim").encode())
        with open(_path, "rb") as _f:
            _code_digest.update(_f.read())
log(f"코드 지문(.py 전체 sha256)={_code_digest.hexdigest()}  git commit={commit or '(git 없음)'}")
if os.path.exists("/content/prelim/SOURCE_COMMIT.txt"):
    log("제출 코드 폴더의 출처: " + open("/content/prelim/SOURCE_COMMIT.txt", encoding="utf-8").read().strip())
with open("/content/prelim/scripts/train_mission2_cnn.py", encoding="utf-8") as _f:
    assert "--early-stop-patience" in _f.read(), "브랜치에 속도 개선 코드가 아직 없음 -- push 됐는지 확인할 것"

[2026-10-12 02:18:52] 코드: 제출 코드 폴더 /content/drive/MyDrive/데이터 크리에이터 캠프/제출/mission2
[2026-10-12 02:18:52] $ pip install -q -r requirements.txt
[2026-10-12 02:18:55] $ pip list 2>/dev/null | grep -i -E '^(numpy|pandas|pyarrow|librosa|scikit-learn|lightgbm|tqdm|torch) '
librosa                               0.11.0
lightgbm                              4.6.0
numpy                                 2.1.3
pandas                                2.2.3
pyarrow                               23.0.1
scikit-learn                          1.6.1
torch                                 2.11.0+cu130
tqdm                                  4.67.3
[2026-10-12 02:18:57] 코드 지문(.py 전체 sha256)=c12c06571fd7b90c4d171f213c032b2d18582ba13590831942f86cb7e326566a  git commit=(git 없음)
[2026-10-12 02:18:57] 제출 코드 폴더의 출처: eigensinn-dcc2026/prelim @ 9a8e266f294e06719b20925a3f395eb98e61562d


## 3-2. 데이터 준비: 원본 zip → 음성 풀기 → 스펙트로그램 → 하나로 묶기

- 약 1~1.5시간 예상. 결과(`m2_spectrograms_packed.npy` 약 7GB + 인덱스)는 이번 실행의 Drive 폴더(`OUT_DIR`)에 저장되고,
  같은 실행이 끊겼다 이어질 때만 복사해서 다시 쓴다.
- Training 전체(서울, 29,200통화)만 학습에 쓴다. 학습기가 인덱스의 `split=Training`, `label_source=provided_json`을 확인하고 아니면 멈춘다.

In [4]:
# 3-2. 데이터 준비
import pandas as pd

PACKED_LOCAL = f"/content/{PACKED_NAME}"
PACKED_INDEX_LOCAL = f"/content/{PACKED_INDEX_NAME}"
PACKED_DRIVE = f"{OUT_DIR}/{PACKED_NAME}"
PACKED_INDEX_DRIVE = f"{OUT_DIR}/{PACKED_INDEX_NAME}"

if os.path.exists(PACKED_DRIVE) and os.path.exists(PACKED_INDEX_DRIVE):
    log(f"이번 실행({RUN_ID})이 앞서 만든 데이터를 다시 복사 (런타임이 끊겨 /content 가 비었음 -- 만든 기록은 이 로그 위쪽에 있음)")
    for src, dst in [(PACKED_INDEX_DRIVE, PACKED_INDEX_LOCAL), (PACKED_DRIVE, PACKED_LOCAL)]:
        if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
            copy_atomic(src, dst)
else:
    assert os.path.isdir(ZIP_DIR), f"원본 zip 폴더가 없음: {ZIP_DIR} -- Drive 경로 확인"
    log("원본 zip 에서 서울 Training 전체 + Validation 전체 압축 풀기")
    run(f'python scripts/extract_dcc_sample_from_zips.py --zip-dir "{ZIP_DIR}" --output-dir {DATA_DIR} --n-train-calls 0')
    run(f"python scripts/build_manifest.py --data-root {DATA_DIR} --output /content/dcc_manifest.csv --overwrite")
    log("발화별 log-mel 스펙트로그램 만들기 (40 mel, hop 160, FFT 512 = 91.55% 모델과 같은 설정)")
    run(
        "python scripts/build_mission2_spectrograms.py --manifest /content/dcc_manifest.csv "
        f"--data-root {DATA_DIR} --split training --n-calls 0 --workers {os.cpu_count()} "
        f"--output-dir {SPEC_DIR} --index-output /content/m2_spectrogram_index.parquet "
        f'--failures-log "{OUT_DIR}/m2_spectrogram_build_failures.jsonl"'
    )
    log("스펙트로그램을 float16 배열 하나로 묶기")
    run(
        "python scripts/pack_mission2_spectrograms.py --index /content/m2_spectrogram_index.parquet "
        f"--spectrogram-dir {SPEC_DIR} --packed-output {PACKED_LOCAL} --index-output {PACKED_INDEX_LOCAL}"
    )
    shutil.rmtree(SPEC_DIR)  # 묶었으니 발화별 파일(~14GB)은 지운다
    log("Drive 에 저장 (이 실행이 끊겼을 때 이어가기용)")
    copy_atomic(PACKED_INDEX_LOCAL, PACKED_INDEX_DRIVE)
    copy_atomic(PACKED_LOCAL, PACKED_DRIVE)
    log(f"packed sha256={sha256(PACKED_LOCAL)}")

_idx = pd.read_parquet(PACKED_INDEX_LOCAL)
log(f"학습 데이터: {len(_idx):,}개 발화 / {_idx['call_id'].nunique():,}통화 "
    f"(91.55% 모델 때: 889,358 / 29,200) split={sorted(_idx['split'].unique())} "
    f"label_source={sorted(_idx['label_source'].unique())} n_mels={sorted(_idx['n_mels'].unique())}")
assert set(_idx["split"]) == {"Training"} and set(_idx["label_source"]) == {"provided_json"}
if (len(_idx), _idx["call_id"].nunique()) != (889_358, 29_200):
    log("⚠️ 발화/통화 수가 예전과 다름 -- 원본 zip 이 다 풀렸는지 확인할 것 (학습은 계속 가능)")

[2026-10-12 02:18:57] 이번 실행(rerun_2026-10-06b)이 앞서 만든 데이터를 다시 복사 (런타임이 끊겨 /content 가 비었음 -- 만든 기록은 이 로그 위쪽에 있음)
[2026-10-12 02:22:03] 학습 데이터: 889,358개 발화 / 29,200통화 (91.55% 모델 때: 889,358 / 29,200) split=['Training'] label_source=['provided_json'] n_mels=[np.int64(40)]


## 4. 학습 (시드마다 1에폭씩, Drive에 매 에폭 저장, 자동 멈춤)

- 에폭마다 `epoch N train_loss=… val_acc=… lr=… time=…s` 한 줄이 셀과 로그 파일에 남는다.
- `val_acc`는 Training 안에서 떼어 둔 20% 통화(배치 패딩 기준)이고, 공식 Validation은 학습에 쓰지 않는다.
- 끊기면 1번 셀부터 다시 실행하면 Drive의 마지막 에폭부터 이어서 한다(옵티마이저, 학습률 스케줄, 난수 상태까지 그대로).

In [5]:
# 4. 학습
def drive_ckpt(seed: int) -> str:
    return f"{OUT_DIR}/m2_cnn_large_rerun_ts{seed}.pt"


def ckpt_status(path: str) -> tuple[int, int, float]:
    if not os.path.exists(path):
        return 0, 0, float("nan")
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    return checkpoint["total_epochs_trained"], checkpoint["best_epoch"], checkpoint["best_val_acc"]


def train_seed(seed: int) -> None:
    local = f"{LOCAL_CKPT_DIR}/m2_cnn_large_rerun_ts{seed}.pt"
    if os.path.exists(drive_ckpt(seed)) and ckpt_status(drive_ckpt(seed))[0] > ckpt_status(local)[0]:
        shutil.copyfile(drive_ckpt(seed), local)
    while True:
        trained, best_epoch, best_acc = ckpt_status(local)
        if trained >= MAX_EPOCHS:
            log(f"[ts{seed}] 끝: {MAX_EPOCHS}에폭 도달 (best epoch {best_epoch}, val_acc {best_acc:.4f})")
            return
        if trained and trained - best_epoch >= EARLY_STOP_PATIENCE:
            log(f"[ts{seed}] 끝: 조기 종료 -- {trained}에폭까지 학습, best epoch {best_epoch} 이후 "
                f"{trained - best_epoch}에폭 개선 없음 (val_acc {best_acc:.4f})")
            return
        log(f"[ts{seed}] {trained + 1}번째 에폭 학습")
        run(
            "python scripts/train_mission2_cnn.py "
            f"--index {PACKED_INDEX_LOCAL} --packed-spectrograms {PACKED_LOCAL} "
            f"--model large --device cuda --seed 42 --torch-seed {seed} "
            "--amp --pad-to-multiple 8 --cudnn-benchmark --num-workers 2 "
            f"--epochs 1 --early-stop-patience {EARLY_STOP_PATIENCE} --model-output {local}"
            + (" --resume" if trained else "")
        )
        copy_atomic(local, drive_ckpt(seed))  # 매 에폭 Drive 에 저장


for _seed in SEEDS:
    train_seed(_seed)

[2026-10-12 02:22:05] [ts1] 끝: 조기 종료 -- 22에폭까지 학습, best epoch 17 이후 5에폭 개선 없음 (val_acc 0.9143)
[2026-10-12 02:22:06] [ts2] 끝: 조기 종료 -- 24에폭까지 학습, best epoch 19 이후 5에폭 개선 없음 (val_acc 0.9143)
[2026-10-12 02:22:07] [ts3] 끝: 조기 종료 -- 20에폭까지 학습, best epoch 15 이후 5에폭 개선 없음 (val_acc 0.9150)


In [6]:
# 5. 결과 요약 + 앙상블로 묶기
for _seed in SEEDS:
    if not os.path.exists(drive_ckpt(_seed)):
        log(f"ts{_seed}: 아직 없음")
        continue
    trained, best_epoch, best_acc = ckpt_status(drive_ckpt(_seed))
    log(f"ts{_seed}: {trained}에폭 학습, best epoch {best_epoch}, 내부 검증 val_acc {best_acc:.4f}, "
        f"sha256={sha256(drive_ckpt(_seed))}")

ENSEMBLE = f"{OUT_DIR}/m2_cnn_large_rerun_ensemble.pt"
_done = [s for s in SEEDS if os.path.exists(drive_ckpt(s))]
_finished = all(
    (lambda t, b, _a: t >= MAX_EPOCHS or t - b >= EARLY_STOP_PATIENCE)(*ckpt_status(drive_ckpt(s))) for s in _done
)
assert len(_done) == len(SEEDS) and _finished, "학습이 다 끝나지 않음 -- 4번 셀부터 다시 실행"
run("python scripts/pack_mission2_l2_ensemble.py "
    + " ".join(f'--ckpt-path "{drive_ckpt(s)}"' for s in SEEDS) + f' --output "{ENSEMBLE}"')
log(f"앙상블 sha256={sha256(ENSEMBLE)}")

[2026-10-12 02:22:07] ts1: 22에폭 학습, best epoch 17, 내부 검증 val_acc 0.9143, sha256=dcaec91e138e6962f62bd5331550bb184a169f2f78da7e8b89b39f83731d0fd5
[2026-10-12 02:22:07] ts2: 24에폭 학습, best epoch 19, 내부 검증 val_acc 0.9143, sha256=19ddcfc24daafa02366744de0e6e4043ca7f37e338ff4d73518ad8a7bfb90d89
[2026-10-12 02:22:07] ts3: 20에폭 학습, best epoch 15, 내부 검증 val_acc 0.9150, sha256=dacf6c335419c502a7de409f762230dc404a9ec0fb22ce5556330a094416808e
[2026-10-12 02:22:08] $ python scripts/pack_mission2_l2_ensemble.py --ckpt-path "/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_rerun_from_raw/rerun_2026-10-06b/m2_cnn_large_rerun_ts1.pt" --ckpt-path "/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_rerun_from_raw/rerun_2026-10-06b/m2_cnn_large_rerun_ts2.pt" --ckpt-path "/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_rerun_from_raw/rerun_2026-10-06b/m2_cnn_large_rerun_ts3.pt" --output "/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_rerun_from_raw/rerun_2026-10-06b/m2_cnn_large_rerun_ensemble.pt"
/content/dr

## 6. (선택) 공식 Validation 평가 — 제출용 `mission2_inference.py` 그대로, 발화 하나씩

CPU로 발화 하나씩 돌리므로 몇 시간 걸릴 수 있다(로컬 PC에서는 약 1.5시간이었음). 최종 결과까지 이 노트북에 남기려면 실행하는 것을 권장.
공식 Validation은 평가에만 쓴다(학습/모델 선택에 쓰지 않음).

In [9]:
# 6. 공식 Validation 평가 (선택)
import csv, json

def find_validation_root():
    """DATA_DIR 아래에서 1.원천데이터/2.라벨링데이터 를 가진 Validation 폴더를 찾는다. 전체 추출(3-2)은 zip 안의
    바깥 폴더 이름을 떼고 풀지만, Validation 만 다시 푸는 추출(아래)은 그 이름을 남겨서 깊이가 다르다."""
    for root, dirs, _files in os.walk(DATA_DIR):
        if os.path.basename(root) == "Validation" and {"1.원천데이터", "2.라벨링데이터"} <= set(dirs):
            return root
    return None


if find_validation_root() is None:  # 런타임이 끊겨 /content 가 비었으면 Validation 만 다시 푼다
    run(f'python scripts/extract_dcc_sample_from_zips.py --zip-dir "{ZIP_DIR}" --output-dir {DATA_DIR} '
        "--n-train-calls 1 --n-val-calls 1000000 --seed 42")
VAL_ROOT = find_validation_root()
assert VAL_ROOT, f"{DATA_DIR} 아래에서 Validation 폴더를 못 찾음"
VAL_AUDIO = f"{VAL_ROOT}/1.원천데이터"
VAL_LABEL = f"{VAL_ROOT}/2.라벨링데이터"
log(f"공식 Validation 폴더: {VAL_ROOT}")
PRED_CSV = f"{OUT_DIR}/official_validation_predictions_rerun_ensemble.csv"
# 채점과 같은 명령: 제출 폴더의 inference.py (GitHub 로 받은 경우엔 같은 구현인 scripts/mission2_inference.py)
_entry = "inference.py" if os.path.exists("/content/prelim/inference.py") else "scripts/mission2_inference.py"
run(f'python {_entry} --audio_dir "{VAL_AUDIO}" --label_dir "{VAL_LABEL}" '
    f'--ckpt_path "{ENSEMBLE}" --output "{PRED_CSV}"')

sys.path.insert(0, "/content/prelim/scripts")
from mission2_inference import pair_by_stem_strict

with open(PRED_CSV, encoding="utf-8") as stream:
    predictions = list(csv.DictReader(stream))
row = 0
counts = {(t, p): 0 for t in (0, 1) for p in (0, 1)}
for _stem, wav_path, json_path in pair_by_stem_strict(VAL_AUDIO, VAL_LABEL):
    with open(json_path, encoding="utf-8") as stream:
        utterances = json.load(stream)["utterances"]
    for utterance in utterances:
        predicted = predictions[row]
        assert predicted["audio_file_name"] == wav_path.name and str(predicted["startAt"]) == str(utterance["startAt"])
        counts[(int(utterance["speaker"]), int(predicted["speaker"]))] += 1
        row += 1
assert row == len(predictions)
total = sum(counts.values())
correct = counts[(0, 0)] + counts[(1, 1)]
log(f"공식 Validation: {correct:,}/{total:,} = {correct / total:.4%} "
    f"(recall 대원(0)={counts[(0, 0)] / (counts[(0, 0)] + counts[(0, 1)]):.4f}, "
    f"신고자(1)={counts[(1, 1)] / (counts[(1, 0)] + counts[(1, 1)]):.4f}) -- 이전 앙상블 91.55% (102,485/111,947)")

[2026-10-12 02:47:13] 공식 Validation 폴더: /content/dcc_data_full/대학부 데이터/Validation
[2026-10-12 02:47:13] $ python inference.py --audio_dir "/content/dcc_data_full/대학부 데이터/Validation/1.원천데이터" --label_dir "/content/dcc_data_full/대학부 데이터/Validation/2.라벨링데이터" --ckpt_path "/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_rerun_from_raw/rerun_2026-10-06b/m2_cnn_large_rerun_ensemble.pt" --output "/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_rerun_from_raw/rerun_2026-10-06b/official_validation_predictions_rerun_ensemble.csv"
Wrote 111947 rows to /content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_rerun_from_raw/rerun_2026-10-06b/official_validation_predictions_rerun_ensemble.csv (0 fell back to majority class)
[2026-10-12 04:14:21] 공식 Validation: 102,419/111,947 = 91.4888% (recall 대원(0)=0.8715, 신고자(1)=0.9551) -- 이전 앙상블 91.55% (102,485/111,947)


In [10]:
# 6-2. 계산 효율성 정보 (대회 권장 4가지: 파라미터 수, 컴퓨팅 환경, Validation 추론 batch size, 추론 시간)
import re

sys.path.insert(0, "/content/prelim/src")
from dcc_data.mission2_cnn import SpeakerCNNLarge

_ensemble = torch.load(ENSEMBLE, map_location="cpu", weights_only=False)
_n_models = len(_ensemble["ensemble_model_state_dicts"])
_per_model = sum(p.numel() for p in SpeakerCNNLarge().parameters())
with open("/proc/cpuinfo", encoding="utf-8") as _f:
    _cpu = next(line.split(":", 1)[1].strip() for line in _f if line.startswith("model name"))

# 추론 시간 = 로그에서 마지막 `python inference.py` 시작 시각 ~ 바로 다음 "공식 Validation:" 결과 줄 시각 (채점 몇 초 포함)
with open(LOG_FILE, encoding="utf-8") as _f:
    _lines = _f.read().splitlines()
_stamp = re.compile(r"^\[(\d{4}-\d\d-\d\d \d\d:\d\d:\d\d)\] ")
_start = _end = _n_utt = None
for _line in _lines:
    _m = _stamp.match(_line)
    if not _m:
        continue
    if "$ python inference.py" in _line or "$ python scripts/mission2_inference.py" in _line:
        _start, _end = _m.group(1), None
    elif _start and "공식 Validation:" in _line:
        _end = _m.group(1)
        _n_utt = int(re.search(r"/([\d,]+) =", _line).group(1).replace(",", ""))
assert _start and _end, "로그에 공식 Validation 추론 기록이 없음 -- 6번 셀을 먼저 실행"
_seconds = (datetime.datetime.fromisoformat(_end) - datetime.datetime.fromisoformat(_start)).total_seconds()

log("계산 효율성 정보")
log(f"  (1) 파라미터: 모델 1개 {_per_model:,}개 x {_n_models}개 = Total {_per_model * _n_models:,}개, "
    f"Active {_per_model * _n_models:,}개 (발화마다 {_n_models}개 모델을 모두 실행)")
log(f"  (2) 컴퓨팅 환경: 학습 GPU {torch.cuda.get_device_name(0)} (혼합 정밀도) / "
    f"추론 CPU {_cpu} {os.cpu_count()}개 (GPU 미사용), torch {torch.__version__}")
log("  (3) Validation 추론 batch size: 1 (발화 하나씩 따로 추론)")
log(f"  (4) Validation 전체 추론 시간: {_seconds / 60:.1f}분 ({_n_utt:,}개 발화, 발화당 평균 {_seconds / _n_utt * 1000:.1f}ms, "
    "음성 읽기와 스펙트로그램 계산 포함)")

[2026-10-12 04:14:21] 계산 효율성 정보
[2026-10-12 04:14:21]   (1) 파라미터: 모델 1개 421,954개 x 3개 = Total 1,265,862개, Active 1,265,862개 (발화마다 3개 모델을 모두 실행)
[2026-10-12 04:14:21]   (2) 컴퓨팅 환경: 학습 GPU Tesla T4 (혼합 정밀도) / 추론 CPU Intel(R) Xeon(R) CPU @ 2.00GHz 2개 (GPU 미사용), torch 2.11.0+cu130
[2026-10-12 04:14:21]   (3) Validation 추론 batch size: 1 (발화 하나씩 따로 추론)
[2026-10-12 04:14:21]   (4) Validation 전체 추론 시간: 87.1분 (111,947개 발화, 발화당 평균 46.7ms, 음성 읽기와 스펙트로그램 계산 포함)


In [11]:
# 7. 로그 파일 전체 출력 -- 끊겼다 이어 돌린 경우에도 전체 학습 기록이 이 노트북에 남게 한다 (출력 지우기 금지)
with open(LOG_FILE, encoding="utf-8") as stream:
    print(stream.read())

[2026-10-05 16:34:37] ================================================================================
[2026-10-05 16:34:37] 새 실행 시작: RUN_ID=rerun_2026-10-06b SEEDS=[1, 2, 3] MAX_EPOCHS=30 EARLY_STOP_PATIENCE=5
[2026-10-05 16:34:37] GPU=Tesla T4 torch=2.11.0+cu130 CPU=2개 디스크 여유=207GB
[2026-10-05 16:34:50] 코드: 제출 코드 폴더 /content/drive/MyDrive/데이터 크리에이터 캠프/제출/mission2
[2026-10-05 16:34:50] $ pip install -q -r requirements.txt
[2026-10-05 16:34:53] $ pip list 2>/dev/null | grep -i -E '^(numpy|pandas|pyarrow|librosa|scikit-learn|lightgbm|tqdm|torch) '
librosa                               0.11.0
lightgbm                              4.6.0
numpy                                 2.1.3
pandas                                2.2.3
pyarrow                               23.0.1
scikit-learn                          1.6.1
torch                                 2.11.0+cu130
tqdm                                  4.67.3
[2026-10-05 16:34:55] 코드 지문(.py 전체 sha256)=c12c06571fd7b90c4d171f213c032b2d18582ba135